# 04 Figures and Tables

Repository notebook for assembling manuscript-ready figures and tables from generated SACU outputs.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")

def read_optional_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing file: {path}")
        return pd.DataFrame()
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() == ".tsv":
        return pd.read_csv(path, sep="\t")
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported table format: {path.suffix}")

def find_files(patterns, roots):
    rows = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pattern in patterns:
            for path in root.rglob(pattern):
                rows.append({
                    "file": path.name,
                    "path": str(path.relative_to(PROJECT_ROOT)) if PROJECT_ROOT in path.parents else str(path),
                    "size_kb": round(path.stat().st_size / 1024, 2),
                    "modified": pd.to_datetime(path.stat().st_mtime, unit="s"),
                })
    return pd.DataFrame(rows).sort_values("modified", ascending=False).reset_index(drop=True) if rows else pd.DataFrame()

import matplotlib.pyplot as plt

## Locate candidate figure and table files

In [ ]:
figure_files = find_files(
    patterns=["*.png", "*.jpg", "*.jpeg", "*.pdf", "*.svg"],
    roots=[PROJECT_ROOT / "outputs", PROJECT_ROOT / "results"],
)

table_files = find_files(
    patterns=["*.csv", "*.xlsx"],
    roots=[PROJECT_ROOT / "outputs", PROJECT_ROOT / "results"],
)

figure_files, table_files.head(30)

## Build manuscript table index

In [ ]:
table_keywords = {
    "Baseline model performance": "baseline|Stage2C",
    "SACU fusion performance": "fusion|sacu|Stage2D",
    "Clinical operating points": "threshold|operating|Stage2E",
    "Ablation analysis": "ablation|Stage2F",
    "Bootstrap confidence intervals": "bootstrap|confidence|Stage2G",
    "Interpretability analysis": "interpretability|pathway|Stage2I",
    "Coordination ablation": "coordination|Stage2J",
}

rows = []
for table_name, pattern in table_keywords.items():
    matches = table_files[
        table_files["file"].str.contains(pattern, case=False, regex=True, na=False)
    ] if not table_files.empty else pd.DataFrame()
    rows.append({
        "recommended_table": table_name,
        "n_candidate_files": len(matches),
        "primary_candidate": matches.iloc[0]["path"] if len(matches) else None,
    })

manuscript_table_index = pd.DataFrame(rows)
manuscript_table_index

## Build manuscript figure index

In [ ]:
figure_keywords = {
    "ROC operating points": "roc|operating|Stage2E",
    "Precision-recall operating points": "pr|precision|recall|Stage2E",
    "Bootstrap CI figure": "ci|bootstrap|Stage2G",
    "Pathway contribution figure": "pathway|contribution|interpretability|Stage2I",
    "Coordination ablation figure": "coordination|Stage2J",
}

rows = []
for figure_name, pattern in figure_keywords.items():
    matches = figure_files[
        figure_files["file"].str.contains(pattern, case=False, regex=True, na=False)
    ] if not figure_files.empty else pd.DataFrame()
    rows.append({
        "recommended_figure": figure_name,
        "n_candidate_files": len(matches),
        "primary_candidate": matches.iloc[0]["path"] if len(matches) else None,
    })

manuscript_figure_index = pd.DataFrame(rows)
manuscript_figure_index

## Export table and figure indexes

In [ ]:
export_dir = PROJECT_ROOT / "outputs" / "notebook_exports"
export_dir.mkdir(parents=True, exist_ok=True)

manuscript_table_index.to_csv(export_dir / "manuscript_table_index.csv", index=False)
manuscript_figure_index.to_csv(export_dir / "manuscript_figure_index.csv", index=False)

print(f"Exported indexes to: {export_dir}")

## Preview selected table

In [ ]:
SELECTED_TABLE_PATH = None

if SELECTED_TABLE_PATH is None:
    candidates = manuscript_table_index["primary_candidate"].dropna()
    if len(candidates):
        SELECTED_TABLE_PATH = PROJECT_ROOT / candidates.iloc[0]

selected_table = read_optional_table(Path(SELECTED_TABLE_PATH)) if SELECTED_TABLE_PATH else pd.DataFrame()
selected_table.head(20)

## Save compact publication table, if selected

In [ ]:
if not selected_table.empty:
    compact_table = selected_table.copy()
    numeric_cols = compact_table.select_dtypes(include=[np.number]).columns
    compact_table[numeric_cols] = compact_table[numeric_cols].round(4)
    compact_table.to_csv(export_dir / "selected_publication_table.csv", index=False)
    compact_table.head(20)
else:
    compact_table = pd.DataFrame()
    compact_table